## Hyperparameter Optimization - ASHA

This example will use the ASHA algorithm to optimize the hyperparameters of the Basket Option model. Here we will use *SHERPA* which is a package to support hyperparameter optimization for a number of more advanced algorithms.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from torch.optim import Adam, RMSprop, Adagrad
import sherpa
import tempfile
import os
import shutil

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Read Basket Option Data

In [ ]:
base_filename = 'basket_option100000x10'

In [ ]:
# Basket data stored in 10 files - load and concatenate
basket_data_list = list()
for i in range(0,10):
    new_basket_data = pd.read_csv(base_filename + str(i) + '.csv', index_col=[0])
    basket_data_list.append(new_basket_data) 
    
basket_data = pd.concat(basket_data_list)
basket_data.reset_index(inplace=True)

In [ ]:
drop_list = ['index','errorEstimate', 'samples', 'processTime']
basket_dataset = basket_data.drop(drop_list, axis=1)
basket_dataset.head()

In [ ]:
train_df = basket_dataset[:-20000]
validate_df = basket_dataset[-20000:]

def prepareData(df):
    y = pd.DataFrame(df['optionValue'])
    X = df.drop(['optionValue'], axis=1)
    return X, y

X_train, y_train = prepareData(train_df)
X_val, y_val = prepareData(validate_df)

### Standard Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
standard_scalar = StandardScaler()
X_train = standard_scalar.fit_transform(X_train)
X_val = standard_scalar.transform(X_val)

### Build Model

In [ ]:
def create_optimizer(name, learning_rate, parameters):
    if name == 'Adam':
        opt = Adam(parameters, lr=learning_rate)
    elif name == 'RMSprop':
        opt = RMSprop(parameters, lr=learning_rate)
    else:
        opt = Adagrad(parameters, lr=learning_rate)
    return opt

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, no_layers=1, no_neurons=1, input_shape=28):
        super(FeedForward, self).__init__()
        self.layers = nn.ModuleList()
        
        # Creating the first layer that takes the input
        prev_size = input_shape
        size = no_neurons
        for _ in range(no_layers):
            self.layers.append(nn.Linear(prev_size, size))
            prev_size = size
        self.output_layer = nn.Linear(prev_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        x = self.output_layer(x)
        return x

In [ ]:
testm = FeedForward(5, 256)

In [ ]:
print(testm)

In [ ]:
no_epochs = 30
batch_size = 2048

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train.to_numpy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

val_x = torch.Tensor(X_val).to(device)
val_y = torch.Tensor(y_val.to_numpy()).to(device)
val_dataset = TensorDataset(val_x, val_y)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

### SHERPA Setup
Set up the SHERPA parameters and <code>study</code> object.

In [ ]:
parameters = [sherpa.Continuous('learning_rate', [1e-4, 1e-2], 'log'),
              sherpa.Discrete('hidden_size', [32, 512]),
              sherpa.Discrete('no_layers', [1, 5]),
              sherpa.Choice('optimizer', ['Adam', 'RMSprop', 'Adagrad'])]
algorithm = alg = sherpa.algorithms.SuccessiveHalving(r=1, R=81, eta=3, s=0, max_finished_configs=1)
study = sherpa.Study(parameters=parameters,
                     algorithm=algorithm,
                     lower_is_better=True,
                     dashboard_port=8990)

In [ ]:
model_dir = tempfile.mkdtemp()

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, initial_epoch, epochs, study):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(initial_epoch, epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y.squeeze())
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y.squeeze())
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        study.add_observation(trial=trial, iteration=epoch,
                              objective=test_loss,
                              context={'loss': test_loss})

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
def buildModel(trial):
    
    # Get hyperparameters
    lr = trial.parameters['learning_rate']
    no_layers = trial.parameters['no_layers']
    layer_width = trial.parameters['hidden_size']
    optimizer_name = trial.parameters['optimizer']
    
    model = FeedForward(no_layers, layer_width).to(device)
    optimizer = create_optimizer(optimizer_name, lr, model.parameters())
    loss_fn = nn.MSELoss()
    return model, optimizer, loss_fn

### Run Search
We now run the search, passing in the training and validation data sets,

In [ ]:
for trial in study:
    # Getting number of training epochs
    print(trial.parameters['resource']) 
    initial_epoch = {1: 0, 3: 1, 9: 4, 27: 13, 81: 40 }[trial.parameters['resource']]
    epochs = trial.parameters['resource'] + initial_epoch

    print("-"*100)
    print(f"Trial:\t{trial.id}\nEpochs:\t{initial_epoch} to {epochs}\nParameters:{trial.parameters}\n")

    if trial.parameters['load_from'] == "":
        print(f"Creating new model for trial {trial.id}...\n")

        model, optimizer, loss_fn = buildModel(trial)
    else:
        print(f"Loading model from: ", os.path.join(model_dir, trial.parameters['load_from']), "...\n")
        model, optimizer, loss_fn = buildModel(trial)
        model_state_dict = torch.load(os.path.join(model_dir, trial.parameters['load_from']))
        model.load_state_dict(model_state_dict)

    # Train model
    train_model(model, train_loader, val_loader, loss_fn, optimizer, initial_epoch, epochs, study)

    study.finalize(trial=trial)
    print(f"Saving model at: ", os.path.join(model_dir, trial.parameters['save_to']))
    torch.save(model.state_dict(), os.path.join(model_dir, trial.parameters['save_to']))
    study.save(model_dir)

In [ ]:
study.get_best_result()

In [ ]:
shutil.rmtree(model_dir)